# Paper Example 1: constitutive-state transfer

Recursive state transfer and the instantaneous no-state ablation.

This notebook is self-contained: all model, constitutive, training, evaluation, and plotting code is included below. The accompanying `reference_data.csv` contains the reference history used to inspect and reproduce the published comparison.


## Imports, study definition, and constitutive utilities


In [ ]:
"""Paper Example 1: constitutive-state transfer versus no-state ablation.

The solver uses a homogeneous axial specialization of a three-dimensional
block.  The network represents u(x,t), while strain, Maxwell branch states and
stress are generated by differentiable mechanics.  Reference solutions are
never used in the loss.
"""

from __future__ import annotations

import argparse
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn


DTYPE = torch.float64


@dataclass(frozen=True)
class Study:
    name: str
    control: str
    n_branches: int
    nonisothermal: bool
    duration: float = 20.0
    length: float = 10.0
    area: float = 6.0
    e_inf: float = 500.0
    e_branch_total: float = 500.0


STUDIES = {
    "study1": Study("study1", "traction", 1, False),
    "study2": Study("study2", "displacement", 6, False),
    "study3": Study("study3", "displacement", 6, True),
}


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def branch_parameters(n_branches: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    full_tau = torch.tensor([0.1, 0.5, 2.0, 10.0, 50.0, 200.0], dtype=DTYPE, device=device)
    if n_branches == 1:
        tau = full_tau[2:3]
    elif n_branches == 3:
        tau = full_tau[[0, 2, 4]]
    elif n_branches == 6:
        tau = full_tau
    else:
        raise ValueError(f"unsupported branch count: {n_branches}")
    raw = 1.0 / torch.sqrt(tau)
    modulus = 500.0 * raw / raw.sum()
    return tau, modulus


def loading_history(t: torch.Tensor, study: Study) -> torch.Tensor:
    """Prescribed right-boundary traction (MPa) or displacement (mm)."""
    ramp_end, hold_end, unload_end = 2.0, 12.0, 14.0
    amplitude = 8.0 if study.control == "traction" else 0.20
    up = amplitude * torch.clamp(t / ramp_end, 0.0, 1.0)
    down = amplitude * torch.clamp((unload_end - t) / (unload_end - hold_end), 0.0, 1.0)
    return torch.where(t <= hold_end, up, torch.where(t <= unload_end, down, torch.zeros_like(t)))


def temperature_history(t: torch.Tensor, study: Study) -> torch.Tensor:
    if not study.nonisothermal:
        return torch.full_like(t, 323.0)
    # Cold loading/early hold, heating during the hold, then hot unloading.
    cold, hot = 313.0, 333.0
    heat_start, heat_end = 6.0, 10.0
    fraction = torch.clamp((t - heat_start) / (heat_end - heat_start), 0.0, 1.0)
    return cold + (hot - cold) * fraction


def shift_factor(temperature: torch.Tensor) -> torch.Tensor:
    c1, c2, ea_r = 14.8, 45.6, 27403.3
    t_ref, t_a, t_c = 323.0, 336.0, 317.4
    wlf = torch.pow(10.0, -c1 * (temperature - t_ref) / (c2 + temperature - t_ref))
    arr = torch.exp(ea_r * (1.0 / temperature - 1.0 / t_a))
    return torch.where(temperature > t_c, wlf, arr)


def material_increments(t: torch.Tensor, temperature: torch.Tensor, reduced_time: bool) -> torch.Tensor:
    dt = t[1:] - t[:-1]
    if reduced_time:
        return dt / shift_factor(temperature[1:])
    return dt


def recursive_states(
    strain: torch.Tensor,
    increments: torch.Tensor,
    tau: torch.Tensor,
) -> torch.Tensor:
    """Return q with shape (Nx, Nt, Nb), preserving the autograd graph."""
    nx, nt = strain.shape
    q_now = torch.zeros((nx, tau.numel()), dtype=strain.dtype, device=strain.device)
    states = [q_now]
    for n in range(nt - 1):
        ratio = increments[n] / tau
        gamma = torch.exp(-ratio)
        beta = torch.where(torch.abs(ratio) > 1e-12, -torch.expm1(-ratio) / ratio, torch.ones_like(ratio))
        de = strain[:, n + 1 : n + 2] - strain[:, n : n + 1]
        q_now = gamma.unsqueeze(0) * q_now + beta.unsqueeze(0) * de
        states.append(q_now)
    return torch.stack(states, dim=1)


def stress_from_strain(
    strain: torch.Tensor,
    increments: torch.Tensor,
    tau: torch.Tensor,
    modulus: torch.Tensor,
    e_inf: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    q = recursive_states(strain, increments, tau)
    branch_stress = q * modulus.reshape(1, 1, -1)
    stress = e_inf * strain + branch_stress.sum(dim=-1)
    return stress, q, branch_stress


## Neural field and differentiable mechanics


In [ ]:
class FieldNet(nn.Module):
    def __init__(self, width: int = 64, depth: int = 4) -> None:
        super().__init__()
        layers: list[nn.Module] = [nn.Linear(3, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers.extend((nn.Linear(width, width), nn.Tanh()))
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor, t: torch.Tensor, temperature: torch.Tensor, study: Study) -> torch.Tensor:
        xbar = 2.0 * x / study.length - 1.0
        tbar = 2.0 * t / study.duration - 1.0
        tscale = (temperature - 323.0) / 20.0
        raw = self.net(torch.stack((xbar, tbar, tscale), dim=-1)).squeeze(-1)
        position = x / study.length
        if study.control == "displacement":
            # Hard Dirichlet representation: the network learns only the
            # interior correction while both axial boundary values are exact.
            target = loading_history(t, study)
            return position * target + 0.20 * position * (1.0 - position) * raw
        # Traction control: only the zero-displacement end is imposed strongly.
        return position * raw


def make_grid(study: Study, nx: int, nt: int, device: torch.device) -> tuple[torch.Tensor, ...]:
    x_line = torch.linspace(0.0, study.length, nx, dtype=DTYPE, device=device)
    t_line = torch.linspace(0.0, study.duration, nt, dtype=DTYPE, device=device)
    temp_line = temperature_history(t_line, study)
    x = x_line[:, None].expand(nx, nt).clone().requires_grad_(True)
    t = t_line[None, :].expand(nx, nt)
    temp = temp_line[None, :].expand(nx, nt)
    return x_line, t_line, temp_line, x, t, temp


def field_response(
    model: FieldNet,
    study: Study,
    nx: int,
    nt: int,
    device: torch.device,
    n_branches: int | None = None,
    reduced_time: bool = True,
    constitutive_mode: str = "recursive",
) -> dict[str, torch.Tensor]:
    x_line, t_line, temp_line, x, t, temp = make_grid(study, nx, nt, device)
    u = model(x, t, temp, study)
    strain = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
    tau, modulus = branch_parameters(n_branches or study.n_branches, device)
    increments = material_increments(t_line, temp_line, reduced_time)
    if constitutive_mode == "recursive":
        stress, q, branch_stress = stress_from_strain(strain, increments, tau, modulus, study.e_inf)
    elif constitutive_mode == "instantaneous":
        # Conventional pointwise closure: time and temperature are neural
        # coordinates, but no constitutive state is transferred between times.
        q = strain.unsqueeze(-1).expand(-1, -1, tau.numel())
        branch_stress = q * modulus.reshape(1, 1, -1)
        stress = study.e_inf * strain + branch_stress.sum(dim=-1)
    else:
        raise ValueError(f"unsupported constitutive mode: {constitutive_mode}")
    dsigma_dx = torch.autograd.grad(stress.sum(), x, create_graph=True)[0]
    return {
        "x": x,
        "t_line": t_line,
        "temperature": temp_line,
        "u": u,
        "strain": strain,
        "stress": stress,
        "q": q,
        "branch_stress": branch_stress,
        "dsigma_dx": dsigma_dx,
        "tau": tau,
        "modulus": modulus,
    }


def reference_response(study: Study, t: torch.Tensor, n_branches: int, reduced_time: bool) -> dict[str, torch.Tensor]:
    tau, modulus = branch_parameters(n_branches, t.device)
    temperature = temperature_history(t, study)
    increments = material_increments(t, temperature, reduced_time)
    target = loading_history(t, study)
    if study.control == "displacement":
        strain = target / study.length
        stress, q, branch_stress = stress_from_strain(strain[None, :], increments, tau, modulus, study.e_inf)
        return {"strain": strain, "stress": stress[0], "q": q[0], "branch_stress": branch_stress[0]}

    # Solve the same discrete constitutive equation for strain under prescribed stress.
    strain_values = [torch.zeros((), dtype=DTYPE, device=t.device)]
    q_now = torch.zeros(tau.numel(), dtype=DTYPE, device=t.device)
    q_values = [q_now]
    branch_values = [modulus * q_now]
    for n in range(t.numel() - 1):
        ratio = increments[n] / tau
        gamma = torch.exp(-ratio)
        beta = torch.where(torch.abs(ratio) > 1e-12, -torch.expm1(-ratio) / ratio, torch.ones_like(ratio))
        eps_n = strain_values[-1]
        numerator = target[n + 1] - torch.sum(modulus * gamma * q_now) + torch.sum(modulus * beta) * eps_n
        eps_next = numerator / (study.e_inf + torch.sum(modulus * beta))
        q_now = gamma * q_now + beta * (eps_next - eps_n)
        strain_values.append(eps_next)
        q_values.append(q_now)
        branch_values.append(modulus * q_now)
    strain = torch.stack(strain_values)
    q = torch.stack(q_values)
    branch_stress = torch.stack(branch_values)
    return {"strain": strain, "stress": target, "q": q, "branch_stress": branch_stress}


## Training, evaluation, and plotting


In [ ]:
def train_one(
    study: Study,
    epochs: int,
    nx: int,
    nt: int,
    seed: int,
    device: torch.device,
    n_branches: int | None = None,
    constitutive_mode: str = "recursive",
) -> tuple[FieldNet, pd.DataFrame]:
    seed_all(seed)
    model = FieldNet().to(device=device, dtype=DTYPE)
    optimizer = torch.optim.Adam(model.parameters(), lr=2.0e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1), eta_min=2.0e-5)
    records: list[dict[str, float]] = []
    u_scale = 0.20
    sigma_scale = 8.0 if study.control == "traction" else 20.0
    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        response = field_response(
            model,
            study,
            nx,
            nt,
            device,
            n_branches=n_branches,
            reduced_time=True,
            constitutive_mode=constitutive_mode,
        )
        target = loading_history(response["t_line"], study)
        loss_eq = torch.mean((response["dsigma_dx"] / (sigma_scale / study.length)) ** 2)
        loss_ic = torch.mean((response["u"][:, 0] / u_scale) ** 2)
        if study.control == "traction":
            loss_bc = torch.mean(((response["stress"][-1] - target) / sigma_scale) ** 2)
        else:
            loss_bc = torch.mean(((response["u"][-1] - target) / u_scale) ** 2)
        boundary_weight = 50.0 if study.control == "traction" else 10.0
        loss = loss_eq + boundary_weight * loss_bc + loss_ic
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        scheduler.step()
        record = {
            "epoch": epoch,
            "loss": float(loss.detach()),
            "equilibrium": float(loss_eq.detach()),
            "boundary": float(loss_bc.detach()),
            "initial": float(loss_ic.detach()),
            "lr": float(optimizer.param_groups[0]["lr"]),
        }
        records.append(record)
        if epoch == 0 or (epoch + 1) % 500 == 0 or epoch + 1 == epochs:
            print(
                f"[{study.name} {constitutive_mode} Nb={n_branches or study.n_branches}] "
                f"{epoch + 1:5d}/{epochs} "
                f"loss={record['loss']:.4e} eq={record['equilibrium']:.3e} "
                f"bc={record['boundary']:.3e}",
                flush=True,
            )
    return model, pd.DataFrame.from_records(records)


def relative_l2(prediction: np.ndarray, reference: np.ndarray) -> float:
    return float(np.linalg.norm(prediction - reference) / max(np.linalg.norm(reference), 1e-14))


def evaluate_and_save(
    model: FieldNet,
    study: Study,
    output_dir: Path,
    nx: int,
    nt: int,
    device: torch.device,
    n_branches: int,
    suffix: str = "",
    constitutive_mode: str = "recursive",
) -> dict[str, float]:
    response = field_response(
        model,
        study,
        nx,
        nt,
        device,
        n_branches=n_branches,
        reduced_time=True,
        constitutive_mode=constitutive_mode,
    )
    ref = reference_response(study, response["t_line"], n_branches, reduced_time=True)
    physical = reference_response(study, response["t_line"], n_branches, reduced_time=False)
    t = response["t_line"].detach().cpu().numpy()
    pred_u = response["u"][-1].detach().cpu().numpy()
    pred_strain = response["strain"].mean(dim=0).detach().cpu().numpy()
    pred_stress = response["stress"].mean(dim=0).detach().cpu().numpy()
    ref_strain = ref["strain"].detach().cpu().numpy()
    ref_stress = ref["stress"].detach().cpu().numpy()
    temperature = response["temperature"].detach().cpu().numpy()
    table = pd.DataFrame({
        "time_s": t,
        "temperature_K": temperature,
        "predicted_end_displacement_mm": pred_u,
        "reference_end_displacement_mm": ref_strain * study.length,
        "predicted_mean_strain": pred_strain,
        "reference_strain": ref_strain,
        "predicted_mean_stress_MPa": pred_stress,
        "reference_stress_MPa": ref_stress,
        "physical_time_ablation_stress_MPa": physical["stress"].detach().cpu().numpy(),
    })
    branch = response["branch_stress"].mean(dim=0).detach().cpu().numpy()
    for index in range(n_branches):
        table[f"predicted_branch_{index + 1}_stress_MPa"] = branch[:, index]
    name = f"response{suffix}.csv"
    table.to_csv(output_dir / name, index=False)
    metrics = {
        "n_branches": n_branches,
        "constitutive_mode": constitutive_mode,
        "displacement_relative_l2": relative_l2(pred_u, ref_strain * study.length),
        "stress_relative_l2": relative_l2(pred_stress, ref_stress),
        "equilibrium_rms_MPa_per_mm": float(torch.sqrt(torch.mean(response["dsigma_dx"] ** 2)).detach()),
    }
    if study.nonisothermal:
        metrics["physical_time_ablation_stress_relative_l2"] = relative_l2(
            physical["stress"].detach().cpu().numpy(), ref_stress
        )
    return metrics


def plot_study(output_dir: Path, study: Study, responses: list[tuple[int, pd.DataFrame]]) -> None:
    plt.rcParams.update({"font.family": "serif", "font.size": 12, "axes.linewidth": 1.0})
    if study.name == "study2":
        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
        for nb, frame in responses:
            axes[0].plot(frame.time_s, frame.predicted_mean_stress_MPa, lw=2, label=fr"$N_b={nb}$")
            axes[0].plot(frame.time_s, frame.reference_stress_MPa, color="black", lw=0.8, alpha=0.25)
        six = responses[-1][1]
        instantaneous = (study.e_inf + study.e_branch_total) * six.reference_strain
        axes[0].plot(
            six.time_s,
            instantaneous,
            color="#c44e52",
            ls=":",
            lw=2,
            label="No state transfer",
        )
        branch_columns = [column for column in six if column.startswith("predicted_branch_")]
        for column in branch_columns:
            axes[1].plot(six.time_s, six[column], lw=1.5, label=column.split("_")[2])
        axes[0].set(xlabel="Physical time (s)", ylabel="Axial stress (MPa)")
        axes[1].set(xlabel="Physical time (s)", ylabel="Branch stress (MPa)")
        axes[0].legend(frameon=False)
        axes[1].legend(title="Branch", ncol=2, frameon=False)
    else:
        fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.0))
        _, frame = responses[0]
        axes[0].plot(frame.time_s, frame.reference_end_displacement_mm, "k-", lw=2, label="Reference")
        axes[0].plot(frame.time_s, frame.predicted_end_displacement_mm, "--", lw=2, label="RT-RPINN")
        axes[1].plot(frame.time_s, frame.reference_stress_MPa, "k-", lw=2, label="Reference")
        axes[1].plot(frame.time_s, frame.predicted_mean_stress_MPa, "--", lw=2, label="RT-RPINN")
        if "conventional_end_displacement_mm" in frame:
            axes[0].plot(
                frame.time_s,
                frame.conventional_end_displacement_mm,
                ":",
                lw=2,
                color="#c44e52",
                label="Conventional PINN",
            )
            axes[1].plot(
                frame.time_s,
                frame.conventional_mean_stress_MPa,
                ":",
                lw=2,
                color="#c44e52",
                label="Conventional PINN",
            )
        if study.nonisothermal:
            axes[1].plot(frame.time_s, frame.physical_time_ablation_stress_MPa, ":", lw=2, label="Physical-time update")
            axes[2].plot(frame.time_s, frame.temperature_K, color="#c44e52", lw=2)
            axes[2].set(xlabel="Physical time (s)", ylabel="Temperature (K)")
        else:
            branch_columns = [column for column in frame if column.startswith("predicted_branch_")]
            for column in branch_columns:
                axes[2].plot(frame.time_s, frame[column], lw=2)
            axes[2].set(xlabel="Physical time (s)", ylabel="Branch stress (MPa)")
        axes[0].set(xlabel="Physical time (s)", ylabel="End displacement (mm)")
        axes[1].set(xlabel="Physical time (s)", ylabel="Axial stress (MPa)")
        axes[0].legend(frameon=False)
        axes[1].legend(frameon=False)
    for axis in axes:
        axis.tick_params(direction="in", top=True, right=True)
    fig.tight_layout()
    fig.savefig(output_dir / f"{study.name}_summary.png", dpi=300, bbox_inches="tight")
    fig.savefig(output_dir / f"{study.name}_summary.jpg", dpi=300, bbox_inches="tight")
    plt.close(fig)


## Example driver


In [ ]:
def run(args: argparse.Namespace) -> None:
    study = STUDIES[args.study]
    output_dir = Path(args.output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device(args.device if args.device else ("cuda" if torch.cuda.is_available() else "cpu"))
    branch_counts = [1, 3, 6] if study.name == "study2" else [study.n_branches]
    all_metrics: list[dict[str, float]] = []
    frames: list[tuple[int, pd.DataFrame]] = []
    for nb in branch_counts:
        model, history = train_one(
            study,
            args.epochs,
            args.nx,
            args.nt,
            args.seed,
            device,
            n_branches=nb,
            constitutive_mode="recursive",
        )
        suffix = f"_nb{nb}" if len(branch_counts) > 1 else ""
        history.to_csv(output_dir / f"loss_history{suffix}.csv", index=False)
        torch.save(model.state_dict(), output_dir / f"model{suffix}.pth")
        metrics = evaluate_and_save(
            model,
            study,
            output_dir,
            args.nx_eval,
            args.nt_eval,
            device,
            nb,
            suffix=suffix,
            constitutive_mode="recursive",
        )
        metrics.update({"study": study.name, "seed": args.seed, "epochs": args.epochs, "device": str(device)})
        all_metrics.append(metrics)
        frame = pd.read_csv(output_dir / f"response{suffix}.csv")
        if study.name == "study1":
            conventional_model, conventional_history = train_one(
                study,
                args.epochs,
                args.nx,
                args.nt,
                args.seed,
                device,
                n_branches=nb,
                constitutive_mode="instantaneous",
            )
            conventional_history.to_csv(output_dir / "loss_history_instantaneous.csv", index=False)
            torch.save(conventional_model.state_dict(), output_dir / "model_instantaneous.pth")
            conventional_metrics = evaluate_and_save(
                conventional_model,
                study,
                output_dir,
                args.nx_eval,
                args.nt_eval,
                device,
                nb,
                suffix="_instantaneous",
                constitutive_mode="instantaneous",
            )
            conventional_metrics.update(
                {"study": study.name, "seed": args.seed, "epochs": args.epochs, "device": str(device)}
            )
            all_metrics.append(conventional_metrics)
            conventional_frame = pd.read_csv(output_dir / "response_instantaneous.csv")
            frame["conventional_end_displacement_mm"] = conventional_frame.predicted_end_displacement_mm
            frame["conventional_mean_stress_MPa"] = conventional_frame.predicted_mean_stress_MPa
            frame.to_csv(output_dir / f"response{suffix}.csv", index=False)
        frames.append((nb, frame))
    pd.DataFrame(all_metrics).to_csv(output_dir / "metrics.csv", index=False)
    plot_study(output_dir, study, frames)
    metadata = vars(args) | {"resolved_device": str(device), "torch_version": torch.__version__}
    (output_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(pd.DataFrame(all_metrics).to_string(index=False), flush=True)


## Load the bundled data and run the example


In [ ]:
# Notebook configuration
REFERENCE_DATA = pd.read_csv(Path("reference_data.csv"))
display(REFERENCE_DATA.head())

config = argparse.Namespace(
    study="study1",
    epochs=5000,
    nx=24,
    nt=81,
    nx_eval=51,
    nt_eval=201,
    seed=42,
    device=None,
    output_dir="outputs",
)

# Set this to False when inspecting only the bundled reference data.
RUN_TRAINING = True
if RUN_TRAINING:
    run(config)
